In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

Sat Sep 12 13:05:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
ZIP = '/content/drive/MyDrive/tensors_rgb_packed_k.zip'
!ls -lh "$ZIP"
!mkdir -p /content/data
!unzip -q "$ZIP" -d /content/data
!ls /content/data

-rw------- 1 root root 1.2G Sep  9 11:34 /content/drive/MyDrive/tensors_rgb_packed_k.zip
tensors_rgb_packed


In [ ]:
from google.colab import files
uploaded = files.upload()
!ls *.py

Saving config.py to config.py
config.py  dataset.py  evaluate.py  targets.py	train.py  yolo_stride16.py


In [ ]:
!apt-get install -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /content/akv
!/content/akv/bin/pip install -q --upgrade pip
!/content/akv/bin/pip install -q akida-models==1.14.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
DATA = '/content/data/tensors_rgb_packed'
import re

src = open('config.py').read()
src = re.sub(r"^ROOT = Path\(.*?\)$", "ROOT = Path('/content')", src, flags=re.M)
src = re.sub(r"^TENSOR_DIR = .*$", f"TENSOR_DIR = Path('{DATA}')", src, flags=re.M)
src = re.sub(r"^SPLITS_FILE = .*$", f"SPLITS_FILE = Path('{DATA}/splits.json')", src, flags=re.M)
src = re.sub(r"^RUNS_DIR = .*$", "RUNS_DIR = Path('/content/runs')", src, flags=re.M)
open('config.py','w').write(src)

src = open('dataset.py').read()
src = src.replace(
    'if not npy.exists() or not meta_path.exists():',
    'npz = tensor_dir / f"{clip}_tensors.npz"\n    if not meta_path.exists() or (not npy.exists() and not npz.exists()):'
)
src = src.replace(
    'tensors = np.load(npy, mmap_mode="r")',
    'tensors = np.load(npy, mmap_mode="r") if npy.exists() else np.load(npz)["a"]'
)
open('dataset.py','w').write(src)

!grep -n "ROOT\|TENSOR_DIR\|SPLITS_FILE\|RUNS_DIR\|INPUT_SIZE\|^GRID" config.py

18:ROOT = Path('/content')
19:TENSOR_DIR = Path('/content/data/tensors_rgb_packed')
20:# TENSOR_DIR = ROOT / "Data_new" / "tensors_messy"
21:# TENSOR_DIR = ROOT / "Data_new" / "tensors_rgb_448"
22:SPLITS_FILE = Path('/content/data/tensors_rgb_packed/splits.json')
23:RUNS_DIR = Path('/content/runs')
30:INPUT_SIZE = 224
31:#INPUT_SIZE = 448
35:GRID = 14
36:CELL = INPUT_SIZE / GRID  # 32 px either way
41:SCALE = min(INPUT_SIZE / SRC_W, INPUT_SIZE / SRC_H)     # 0.70 at 448
42:PAD_X = (INPUT_SIZE - SRC_W * SCALE) / 2                # 0.0
43:PAD_Y = (INPUT_SIZE - SRC_H * SCALE) / 2                # 44.8


In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u dataset.py


tensors : /content/data/tensors_rgb_packed
policy  : keep

TRAIN  (90 clips in split)
  clips loaded : 90
  samples      : 28045
  quiet frames : 0 (policy: keep)
  boxes        : 28345
  box size     : median 10.4 px (0.65 cells), p5 6.7, p95 17.7
  under 8 px   : 16.4%

VALIDATION  (12 clips in split)
  clips loaded : 12
  samples      : 3728
  quiet frames : 0 (policy: keep)
  boxes        : 3728
  box size     : median 10.2 px (0.64 cells), p5 6.5, p95 18.2
  under 8 px   : 25.0%

TEST  (12 clips in split)
  clips loaded : 12
  samples      : 3723
  quiet frames : 0 (policy: keep)
  boxes        : 4243
  box size     : median 10.6 px (0.66 cells), p5 8.1, p95 20.5
  under 8 px   : 4.6%

One batch:
  images shape : (4, 224, 224, 3)  float32
  value range  : 0.00 to 227.00
  zero pixels  : 20.1%
  boxes/frame  : [1, 1, 1, 1]
  out of bounds: 0
  inside padding: 0


In [ ]:
import subprocess, threading, time, os
os.makedirs('/content/drive/MyDrive/drone_runs', exist_ok=True)

def backup():
    while True:
        time.sleep(600)
        subprocess.run('cp -r /content/runs/* /content/drive/MyDrive/drone_runs/ 2>/dev/null', shell=True)

threading.Thread(target=backup, daemon=True).start()

!MPLBACKEND=Agg /content/akv/bin/python -u train.py --epochs 25 --lr 1e-3 --batch_size 64 --name full_rgb_s16

2026-09-12 13:14:38.484871: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789218878.711654    3829 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789218878.774035    3829 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789218879.217432    3829 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789218879.217489    3829 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789218879.217497    3829 computation_placer.cc:177] computation placer alr

In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u evaluate.py --run full_rgb_s16 --split validation

2026-09-12 13:59:09.214522: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789221549.237471   14956 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789221549.244908   14956 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789221549.263446   14956 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789221549.263489   14956 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789221549.263493   14956 computation_placer.cc:177] computation placer alr

In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u evaluate.py --run full_rgb_s16 --split test
!cp -r /content/runs/* /content/drive/MyDrive/drone_runs/
!cd /content && zip -qr full_rgb_s16.zip runs/full_rgb_s16
!cp /content/full_rgb_s16.zip /content/drive/MyDrive/
!ls -lh /content/full_rgb_s16.zip

2026-09-12 13:59:54.258609: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789221594.293114   15900 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789221594.305499   15900 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789221594.332334   15900 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789221594.332369   15900 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789221594.332377   15900 computation_placer.cc:177] computation placer alr